# Student Score Prediction System
## Consolidated Master Project Notebook - Day 12
**AI & ML Virtual Internship - Codomax Digital Solutions**

### Project Overview
This project builds an end-to-end Machine Learning solution to predict a student's examination score based on their daily study hours. It goes through the entire lifecycle of a Data Science project: Data Loading, Cleaning, Exploration (EDA), Visualization, Feature Engineering, Model Training, Model Evaluation, and finally, deploying an interactive Prediction App.

### Project Pipeline:
1. **Environment Setup & Data Import** (Day 4)
2. **Data Cleaning & Preprocessing** (Day 5)
3. **Data Visualization & Exploratory Data Analysis** (Day 6 & 7)
4. **Linear Regression Model Training** (Day 8)
5. **Prediction on Unseen Data** (Day 9)
6. **Model Performance Evaluation** (Day 10)
7. **Interactive Command-Line App Integration** (Day 11)

## Phase 1: Environment Setup & Data Loading
We import standard packages: Pandas for data structures, NumPy for array mathematics, Matplotlib/Seaborn for visual plotting, and Scikit-learn for ML algorithms.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set aesthetic properties for visualization
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 12

# Load dataset with fallback paths for Google Colab or local envs
dataset_path = "student_scores.csv"
if not os.path.exists(dataset_path):
    dataset_path = "/content/student_scores.csv"
    if not os.path.exists(dataset_path):
        dataset_path = "../student_scores.csv"

df = pd.read_csv(dataset_path)
print("✅ Dataset loaded successfully!")
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## Phase 2: Data Cleaning & Preprocessing
We inspect the dataset for any structural anomalies, missing entries (nulls), or duplicates to ensure the dataset is cleaned properly before modeling.

In [ ]:
print("--- Data Information ---")
df.info()

print("\n--- Null Value Count ---")
print(df.isnull().sum())

print("\n--- Duplicate Count ---")
print(f"Duplicate Rows: {df.duplicated().sum()}")

print("\n--- Statistical Distribution ---")
df.describe()

## Phase 3: Exploratory Data Analysis & Visualization
To understand the linear behavior, we plot `Hours_Studied` against `Scores`. A strong upward trend indicates a high linear correlation between the two variables.

In [ ]:
# 1. Scatter Plot
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="Hours_Studied", y="Scores", s=100, color="#1f77b4", edgecolor="black", alpha=0.8)
plt.title("Correlation between Study Hours & Student Scores")
plt.xlabel("Hours Studied")
plt.ylabel("Exam Score (%)")
plt.show()

# 2. Correlation Coefficient Matrix
corr_matrix = df[["Hours_Studied", "Attendance", "Scores"]].corr()
print("\nCorrelation Matrix:")
print(corr_matrix)
print(f"\nCorrelation between Hours Studied and Scores is: {corr_matrix.loc['Hours_Studied', 'Scores']:.4f}")

## Phase 4: Model Building & Training
We frame this as a Simple Linear Regression task:
$$\text{Scores} = \beta_1 \times \text{Hours\_Studied} + \beta_0$$

We split our data into 80% train and 20% test sets using a fixed seed (`random_state=42`) for reproducibility, and fit the OLS (Ordinary Least Squares) line.

In [ ]:
# Select feature and target variables
X = df[["Hours_Studied"]]
y = df["Scores"]

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Instantiate & train model
model = LinearRegression()
model.fit(X_train, y_train)

print("✅ Linear Regression Model trained successfully!")
print(f"Learned Slope (Coefficient) : {model.coef_[0]:.6f}")
print(f"Learned Y-Intercept         : {model.intercept_:.6f}")

## Phase 5: Predictions & Evaluation
We make predictions on the test set and evaluate performance using Mean Absolute Error (MAE), Mean Squared Error (MSE), Root Mean Squared Error (RMSE), and R-squared ($R^2$) Score.

In [ ]:
# Predict
y_pred = model.predict(X_test)

# Compare actual and predicted
comparison_df = pd.DataFrame({
    "Hours Studied": X_test["Hours_Studied"],
    "Actual Score": y_test,
    "Predicted Score": y_pred.round(2)
}).reset_index(drop=True)

print("--- Predictions Comparison Table ---")
print(comparison_df.to_string())

# Metrics calculation
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("\n--- Evaluation Summary ---")
print(f"Mean Absolute Error (MAE)      : {mae:.4f} marks")
print(f"Mean Squared Error (MSE)       : {mse:.4f}")
print(f"Root Mean Squared Error (RMSE) : {rmse:.4f} marks")
print(f"R-squared (R2) Score           : {r2:.4f} ({r2*100:.2f}%)")

### Plotting the Regression Line
Let's visualize how the regression line fits against both the training and testing data.

In [ ]:
plt.figure(figsize=(9, 6))
# Plot training data
plt.scatter(X_train, y_train, color="#2ca02c", label="Training Data", alpha=0.7, s=70)
# Plot testing data
plt.scatter(X_test, y_test, color="#d62728", label="Testing Data", alpha=0.9, s=90, marker='s')
# Plot regression line
x_line = np.linspace(X["Hours_Studied"].min(), X["Hours_Studied"].max(), 100).reshape(-1, 1)
y_line = model.predict(x_line)
plt.plot(x_line, y_line, color="#1f77b4", linewidth=3, label="Regression Line")

plt.title("Student Score Regression Line Fit")
plt.xlabel("Hours Studied")
plt.ylabel("Scores")
plt.legend()
plt.show()

## Phase 6: Prediction App
We implement the interactive score prediction logic. Call the function below to predict custom inputs.

In [ ]:
def predict_student_score(hours):
    if hours < 0:
        raise ValueError("Study hours cannot be negative.")
    elif hours > 24:
        hours = 24.0
        print("⚠️ Notice: Study hours set to max limit of 24 hours per day.")
        
    # Predict using model
    pred_val = model.predict([[hours]])[0]
    # Cap between 0 and 100%
    final_val = max(0.0, min(100.0, pred_val))
    return pred_val, final_val

# Demonstration
demo_hours = 7.5
raw_p, capped_p = predict_student_score(demo_hours)
print(f"For {demo_hours} study hours, Predicted score: {capped_p:.2f}%")

## Conclusion
1. **Strong Fit:** Study Hours correlate strongly with scores ($R \approx 0.98$).
2. **Highly Predictable:** Our linear regression line explains **96.78%** of score variation ($R^2 = 0.9678$).
3. **Practical Utility:** A student studying **7.5 hours** is expected to score **75.44%**.